# 📡 Phase 1 — Collecte de Données Médiatiques Africaines

**Projet : African Media Intelligence AI**  
**Auteur : Esmel Amari (Phil)**  
**Description :** Ce notebook collecte automatiquement des articles de presse africains via des flux RSS, les nettoie et les sauvegarde dans un fichier CSV structuré prêt pour l'analyse NLP.

---

### Sources ciblées
| Source | Langue | Région |
|---|---|---|
| AllAfrica.com | EN/FR | Panafricaine |
| Jeune Afrique | FR | Panafricaine |
| RFI Afrique | FR | Panafricaine |
| Abidjan.net | FR | Côte d'Ivoire |
| BBC Afrique | FR | Panafricaine |
| The Africa Report | EN | Panafricaine |


## 0. Imports & Configuration

In [1]:
import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
import json
import logging
from datetime import datetime
from pathlib import Path

# ── Configuration logging ──────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

# ── Chemins ────────────────────────────────────────────────────────
BASE_DIR   = Path('..').resolve()
RAW_DIR    = BASE_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = RAW_DIR / f"articles_raw_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

print(f"✅ Dossier de sortie : {RAW_DIR}")

✅ Dossier de sortie : C:\Users\E682\Desktop\data\raw


## 1. Définition des Sources RSS

In [2]:
# ── Flux RSS africains ─────────────────────────────────────────────
RSS_SOURCES = [
    {
        "name": "AllAfrica",
        "url": "https://allafrica.com/tools/headlines/rdf/latest/headlines.rdf",
        "langue": "en",
        "region": "Panafrique"
    },
    {
        "name": "RFI Afrique",
        "url": "https://www.rfi.fr/fr/rss-afrique",
        "langue": "fr",
        "region": "Panafrique"
    },
    {
        "name": "BBC Afrique",
        "url": "https://feeds.bbci.co.uk/afrique/rss.xml",
        "langue": "fr",
        "region": "Panafrique"
    },
    {
        "name": "Jeune Afrique",
        "url": "https://www.jeuneafrique.com/feed/",
        "langue": "fr",
        "region": "Panafrique"
    },
    {
        "name": "The Africa Report",
        "url": "https://www.theafricareport.com/feed/",
        "langue": "en",
        "region": "Panafrique"
    },
    {
        "name": "Abidjan.net",
        "url": "https://news.abidjan.net/rss/news.rss",
        "langue": "fr",
        "region": "Côte d'Ivoire"
    },
    {
        "name": "Fratmat.info",
        "url": "https://www.fratmat.info/feed",
        "langue": "fr",
        "region": "Côte d'Ivoire"
    },
    {
        "name": "Le Monde Afrique",
        "url": "https://www.lemonde.fr/afrique/rss_full.xml",
        "langue": "fr",
        "region": "Panafrique"
    }
]

print(f"📋 {len(RSS_SOURCES)} sources configurées")
for s in RSS_SOURCES:
    print(f"  → [{s['langue'].upper()}] {s['name']} ({s['region']})")

📋 8 sources configurées
  → [EN] AllAfrica (Panafrique)
  → [FR] RFI Afrique (Panafrique)
  → [FR] BBC Afrique (Panafrique)
  → [FR] Jeune Afrique (Panafrique)
  → [EN] The Africa Report (Panafrique)
  → [FR] Abidjan.net (Côte d'Ivoire)
  → [FR] Fratmat.info (Côte d'Ivoire)
  → [FR] Le Monde Afrique (Panafrique)


## 2. Fonctions Utilitaires

In [3]:
def clean_html(raw_html: str) -> str:
    """Supprime les balises HTML et nettoie le texte."""
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, 'html.parser')
    text = soup.get_text(separator=' ')
    # Suppression espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def parse_date(entry) -> str:
    """Extrait et formate la date de publication."""
    for date_field in ['published', 'updated', 'created']:
        raw = entry.get(date_field, '')
        if raw:
            try:
                import email.utils
                parsed = email.utils.parsedate_to_datetime(raw)
                return parsed.strftime('%Y-%m-%d %H:%M:%S')
            except Exception:
                return raw[:19]  # fallback brut
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def extract_article_content(url: str, timeout: int = 10) -> str:
    """
    Tente d'extraire le contenu complet de l'article via scraping.
    Retourne une chaîne vide en cas d'échec (le résumé RSS sera utilisé).
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; AfricanMediaBot/1.0)'}
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')

        # Stratégie : chercher les balises sémantiques en priorité
        for selector in ['article', 'main', '.article-content',
                          '.post-content', '#article-body', '.entry-content']:
            block = soup.select_one(selector)
            if block:
                paragraphs = block.find_all('p')
                content = ' '.join(p.get_text() for p in paragraphs)
                if len(content) > 200:
                    return re.sub(r'\s+', ' ', content).strip()

        # Fallback : tous les paragraphes de la page
        paragraphs = soup.find_all('p')
        content = ' '.join(p.get_text() for p in paragraphs[:15])
        return re.sub(r'\s+', ' ', content).strip()

    except Exception as e:
        logger.debug(f"Scraping échoué pour {url}: {e}")
        return ""


def word_count(text: str) -> int:
    """Compte le nombre de mots dans un texte."""
    return len(text.split()) if text else 0


print("✅ Fonctions utilitaires définies")

✅ Fonctions utilitaires définies


## 3. Collecte RSS — Fonction Principale

In [4]:
def collect_from_source(source: dict, scrape_full: bool = False, delay: float = 1.5) -> list:
    """
    Collecte les articles d'un flux RSS.

    Paramètres :
        source      : dict avec name, url, langue, region
        scrape_full : si True, tente de scraper le contenu complet
        delay       : délai entre requêtes (politesse)

    Retourne :
        Liste de dicts (un par article)
    """
    articles = []
    logger.info(f"Collecte : {source['name']} ({source['url']})")

    try:
        feed = feedparser.parse(source['url'])

        if feed.bozo and not feed.entries:
            logger.warning(f"Flux invalide ou inaccessible : {source['name']}")
            return articles

        logger.info(f"  → {len(feed.entries)} entrées trouvées")

        for entry in feed.entries:
            # ── Titre ──────────────────────────────────────
            titre = clean_html(entry.get('title', ''))
            if not titre:
                continue

            # ── Résumé RSS ─────────────────────────────────
            resume = clean_html(
                entry.get('summary', '') or
                entry.get('description', '') or
                entry.get('content', [{}])[0].get('value', '')
            )

            # ── URL ────────────────────────────────────────
            url = entry.get('link', '')

            # ── Contenu complet (optionnel) ────────────────
            contenu_complet = ""
            if scrape_full and url:
                contenu_complet = extract_article_content(url)
                time.sleep(delay)

            # ── Texte principal (résumé ou full content) ───
            texte = contenu_complet if contenu_complet else resume

            # ── Tags / catégories ──────────────────────────
            tags = ', '.join(
                t.get('term', '') for t in entry.get('tags', [])
                if t.get('term')
            )

            article = {
                # ─ Identification ─
                'id'             : f"{source['name'].lower().replace(' ', '_')}_{abs(hash(url)) % 10**8}",
                'source'         : source['name'],
                'langue'         : source['langue'],
                'region'         : source['region'],
                # ─ Contenu ─
                'titre'          : titre,
                'resume'         : resume,
                'contenu'        : texte,
                'url'            : url,
                'tags'           : tags,
                # ─ Temporel ─
                'date_publication': parse_date(entry),
                'date_collecte'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                # ─ Métriques ─
                'nb_mots'        : word_count(texte),
                'scraping_full'  : bool(contenu_complet),
            }
            articles.append(article)

    except Exception as e:
        logger.error(f"Erreur sur {source['name']}: {e}")

    return articles


print("✅ Fonction de collecte définie")

✅ Fonction de collecte définie


## 4. Lancement de la Collecte

In [5]:
# ── Paramètre : activer le scraping complet ou non ─────────────────
# Mettre SCRAPE_FULL = True pour tenter d'extraire le texte intégral
# Mettre False pour utiliser uniquement les résumés RSS (plus rapide)
SCRAPE_FULL = False

all_articles = []
stats = []

for source in RSS_SOURCES:
    articles = collect_from_source(source, scrape_full=SCRAPE_FULL)
    all_articles.extend(articles)
    stats.append({'source': source['name'], 'articles_collectes': len(articles)})
    time.sleep(1)  # Politesse entre sources

print(f"\n{'='*50}")
print(f"✅ Total articles collectés : {len(all_articles)}")
print(f"{'='*50}")
for s in stats:
    print(f"  {s['source']:<25} → {s['articles_collectes']} articles")

2026-05-12 20:15:42,588 [INFO] Collecte : AllAfrica (https://allafrica.com/tools/headlines/rdf/latest/headlines.rdf)
2026-05-12 20:15:44,827 [INFO]   → 30 entrées trouvées
2026-05-12 20:15:45,955 [INFO] Collecte : RFI Afrique (https://www.rfi.fr/fr/rss-afrique)
2026-05-12 20:15:48,796 [WARNING] Flux invalide ou inaccessible : RFI Afrique
2026-05-12 20:15:49,799 [INFO] Collecte : BBC Afrique (https://feeds.bbci.co.uk/afrique/rss.xml)
2026-05-12 20:15:51,552 [INFO]   → 43 entrées trouvées
2026-05-12 20:15:52,581 [INFO] Collecte : Jeune Afrique (https://www.jeuneafrique.com/feed/)
2026-05-12 20:15:54,288 [INFO]   → 30 entrées trouvées
2026-05-12 20:15:55,304 [INFO] Collecte : The Africa Report (https://www.theafricareport.com/feed/)
2026-05-12 20:15:57,086 [INFO]   → 10 entrées trouvées
2026-05-12 20:15:58,099 [INFO] Collecte : Abidjan.net (https://news.abidjan.net/rss/news.rss)
2026-05-12 20:16:00,126 [WARNING] Flux invalide ou inaccessible : Abidjan.net
2026-05-12 20:16:01,127 [INFO] Co


✅ Total articles collectés : 133
  AllAfrica                 → 30 articles
  RFI Afrique               → 0 articles
  BBC Afrique               → 43 articles
  Jeune Afrique             → 30 articles
  The Africa Report         → 10 articles
  Abidjan.net               → 0 articles
  Fratmat.info              → 0 articles
  Le Monde Afrique          → 20 articles


## 5. Création du DataFrame & Nettoyage

In [6]:
df = pd.DataFrame(all_articles)

print(f"📊 Shape brut : {df.shape}")
print(f"\nColonnes :")
print(df.dtypes)

# ── Nettoyage ──────────────────────────────────────────────────────
# 1. Suppression des doublons (même URL)
avant = len(df)
df = df.drop_duplicates(subset=['url'], keep='first')
print(f"\n🔧 Doublons supprimés (URL) : {avant - len(df)}")

# 2. Suppression des doublons (même titre)
avant = len(df)
df = df.drop_duplicates(subset=['titre'], keep='first')
print(f"🔧 Doublons supprimés (titre) : {avant - len(df)}")

# 3. Filtre articles trop courts (< 20 mots → probablement inutiles)
avant = len(df)
df = df[df['nb_mots'] >= 20].copy()
print(f"🔧 Articles trop courts (<20 mots) supprimés : {avant - len(df)}")

# 4. Remplissage des valeurs manquantes
df['tags']    = df['tags'].fillna('')
df['resume']  = df['resume'].fillna('')
df['contenu'] = df['contenu'].fillna('')

# 5. Colonne texte unifié pour NLP (titre + contenu)
df['texte_nlp'] = df['titre'] + '. ' + df['contenu']

# 6. Reset index
df = df.reset_index(drop=True)

print(f"\n✅ DataFrame final : {df.shape[0]} articles, {df.shape[1]} colonnes")

📊 Shape brut : (133, 13)

Colonnes :
id                  object
source              object
langue              object
region              object
titre               object
resume              object
contenu             object
url                 object
tags                object
date_publication    object
date_collecte       object
nb_mots              int64
scraping_full         bool
dtype: object

🔧 Doublons supprimés (URL) : 0
🔧 Doublons supprimés (titre) : 0
🔧 Articles trop courts (<20 mots) supprimés : 12

✅ DataFrame final : 121 articles, 14 colonnes


## 6. Exploration Rapide du Dataset

In [7]:
print("=== APERÇU DES 3 PREMIERS ARTICLES ===")
print(df[['source', 'titre', 'nb_mots', 'date_publication']].head(3).to_string())

print("\n=== DISTRIBUTION PAR SOURCE ===")
print(df['source'].value_counts().to_string())

print("\n=== DISTRIBUTION PAR LANGUE ===")
print(df['langue'].value_counts().to_string())

print("\n=== DISTRIBUTION PAR RÉGION ===")
print(df['region'].value_counts().to_string())

print("\n=== STATISTIQUES NB_MOTS ===")
print(df['nb_mots'].describe().to_string())

=== APERÇU DES 3 PREMIERS ARTICLES ===
      source                                                                                                                       titre  nb_mots     date_publication
0  AllAfrica                                     Kenya: MPs Probe Missing 27,000 Tonnes of Imported Sugar Declared Unfit for Consumption       35  2026-05-13 06:45:46
1  AllAfrica                                                                        Nigeria: Over 100 Killed in Zamfara Market Airstrike       36  2026-05-13 05:08:56
2  AllAfrica  Sudan: UN Warns Sudan War Entering 'Deadlier Phase' As Drone Strikes Kill Hundreds, Attacks Spread Across Multiple Regions       49  2026-05-12 20:51:13

=== DISTRIBUTION PAR SOURCE ===
source
BBC Afrique          36
Jeune Afrique        30
AllAfrica            26
Le Monde Afrique     20
The Africa Report     9

=== DISTRIBUTION PAR LANGUE ===
langue
fr    86
en    35

=== DISTRIBUTION PAR RÉGION ===
region
Panafrique    121

=== STATISTIQUES

## 7. Sauvegarde

In [8]:
# ── Sauvegarde CSV principal ───────────────────────────────────────
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"✅ Sauvegardé : {OUTPUT_FILE}")
print(f"   → {df.shape[0]} articles | {round(OUTPUT_FILE.stat().st_size / 1024, 1)} KB")

# ── Sauvegarde du rapport de collecte ─────────────────────────────
rapport = {
    'date_collecte'      : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_articles'     : int(df.shape[0]),
    'sources'            : df['source'].value_counts().to_dict(),
    'langues'            : df['langue'].value_counts().to_dict(),
    'regions'            : df['region'].value_counts().to_dict(),
    'nb_mots_moyen'      : round(df['nb_mots'].mean(), 1),
    'fichier_sortie'     : str(OUTPUT_FILE.name)
}

rapport_path = RAW_DIR / 'collecte_rapport.json'
with open(rapport_path, 'w', encoding='utf-8') as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)

print(f"✅ Rapport JSON : {rapport_path}")
print("\n📋 Résumé du rapport :")
print(json.dumps(rapport, ensure_ascii=False, indent=2))

✅ Sauvegardé : C:\Users\E682\Desktop\data\raw\articles_raw_20260512_2014.csv
   → 121 articles | 135.3 KB
✅ Rapport JSON : C:\Users\E682\Desktop\data\raw\collecte_rapport.json

📋 Résumé du rapport :
{
  "date_collecte": "2026-05-12 20:17:03",
  "total_articles": 121,
  "sources": {
    "BBC Afrique": 36,
    "Jeune Afrique": 30,
    "AllAfrica": 26,
    "Le Monde Afrique": 20,
    "The Africa Report": 9
  },
  "langues": {
    "fr": 86,
    "en": 35
  },
  "regions": {
    "Panafrique": 121
  },
  "nb_mots_moyen": 37.3,
  "fichier_sortie": "articles_raw_20260512_2014.csv"
}


---

## ✅ Phase 1 Terminée

**Ce qui a été accompli :**
- ✅ Collecte de **8 sources RSS** africaines (FR + EN)
- ✅ Nettoyage HTML, déduplication, filtrage
- ✅ Création de la colonne `texte_nlp` prête pour l'analyse
- ✅ Sauvegarde CSV + rapport JSON

**Prochaine étape :** `02_sentiment_analysis.ipynb` — Analyse de sentiment avec HuggingFace Transformers
